---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## Préparation : journal d'exécution

Les consignes des cellules suivantes demandent de « laisser une trace dans les logs ».
On définit une fonction `journaliser()` qui écrit chaque événement (alerte, injection de données
tardives, arrêt des requêtes) dans un fichier `streaming.log`, en plus de l'affichage.

In [24]:
import logging
import shutil
import subprocess
import sys
from datetime import datetime, timezone

SCRIPTS_DIR = Path("../scripts")
LOG_DIR     = OUTPUT_DIR / "logs"
LOG_STREAM  = LOG_DIR / "streaming.log"
LOG_SIM     = LOG_DIR / "simulateur.log"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Un logger Python standard : horodatage automatique, niveau, et écriture dans un fichier.
logger = logging.getLogger("climacity.streaming")
logger.setLevel(logging.INFO)
if not logger.handlers:   # évite de dupliquer les lignes si la cellule est relancée
    gestionnaire = logging.FileHandler(LOG_STREAM, mode="w", encoding="utf-8")
    gestionnaire.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(gestionnaire)

def journaliser(message: str, niveau: int = logging.INFO) -> None:
    """Écrit un message dans le fichier de logs du streaming et l'affiche.

    Args:
        message: Texte de l'événement.
        niveau: Niveau ``logging`` (INFO par défaut).

    Example:
        >>> journaliser("Requête démarrée")
        [LOG] Requête démarrée
    """
    logger.log(niveau, message)
    print(f"[LOG] {message}")

journaliser(f"Journal de streaming initialisé : {LOG_STREAM}")

[LOG] Journal de streaming initialisé : ../data/output/logs/streaming.log


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


### Lancement du simulateur depuis le carnet

Pour que le carnet s'exécute **de la première à la dernière cellule sans intervention**
(contrainte n°2 de l'énoncé), je démarre le simulateur ici en processus d'arrière-plan
(`subprocess.Popen`) plutôt que dans un terminal séparé. Je repars aussi d'un état propre :
je supprime l'ancien flux, les anciens points de reprise (*checkpoints*) et les anciens résultats,
sans quoi un rejeu additionnerait de vieux résultats aux nouveaux.

Passer `LANCER_SIMULATEUR = False` pour le lancer soi-même dans un terminal, comme indiqué
ci-dessus.

In [25]:
LANCER_SIMULATEUR = True
CHEMIN_FENETRES   = DELTA_DISPONIBLE.parent / "fenetres_arrondissement"
SIM_PROC = None

if LANCER_SIMULATEUR:
    # État propre : flux, checkpoints et résultats d'une exécution précédente
    for dossier in (STREAM_SOURCE_DIR, STREAM_CHECKPOINT, CHEMIN_FENETRES):
        shutil.rmtree(dossier, ignore_errors=True)
    STREAM_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    STREAM_CHECKPOINT.mkdir(parents=True, exist_ok=True)

    commande = [
        sys.executable, str(SCRIPTS_DIR / "simulateur_flux.py"),
        "--input",   str(VELIB_CONSOLIDE),
        "--output",  str(STREAM_SOURCE_DIR),
        "--vitesse", "3",        # 1 s réelle = 3 min de données : un relevé (15 min) toutes les 5 s
    ]
    # Popen lance le processus SANS attendre sa fin : il tourne pendant que le carnet continue.
    SIM_PROC = subprocess.Popen(commande, stdout=open(LOG_SIM, "w"), stderr=subprocess.STDOUT)
    journaliser(f"Simulateur lancé (pid {SIM_PROC.pid}) : {' '.join(commande[1:])}")
else:
    print("Simulateur non lancé : démarrez-le dans un terminal séparé (voir ci-dessus).")

[LOG] Simulateur lancé (pid 16295) : ../scripts/simulateur_flux.py --input ../data/output/disponibilite_consolidee.parquet --output ../data/output/stream_input --vitesse 3


In [26]:
# Vérification : le simulateur tourne-t-il ?
import time, os, json

def compter_fichiers_stream(attente_sec: int = 5) -> int:
    """Attend et compte les fichiers JSON produits par le simulateur.

    Args:
        attente_sec: Temps d'attente avant de compter, en secondes.

    Returns:
        Le nombre de fichiers ``velib_*.json`` présents dans le répertoire surveillé.
    """
    time.sleep(attente_sec)
    # Récupérer la liste des fichiers. Le motif « velib_*.json » exclut les fichiers
    # temporaires (.tmp_...) que le simulateur renomme une fois complets.
    fichiers = sorted(STREAM_SOURCE_DIR.glob("velib_*.json"))
    return len(fichiers)

n = compter_fichiers_stream(3)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print(f"  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    # Aperçu du dernier fichier produit
    # Les noms portent 6 chiffres : l'ordre alphabétique reproduit l'ordre d'émission.
    dernier = max(STREAM_SOURCE_DIR.glob("velib_*.json"))
    contenu = json.loads(dernier.read_text(encoding="utf-8"))
    print(f"{n} fichier(s) présent(s). Dernier : {dernier.name}  ({len(contenu)} relevés de stations)")
    print("Premier relevé :", contenu[0])

[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.
  Vérifiez que le simulateur tourne dans un terminal séparé.
  Répertoire surveillé : ../data/output/stream_input


---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [27]:
import json

# Schéma du flux JSON produit par le simulateur
schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       TimestampType(), False),   # timestamp parsé
])

# Création du DataFrame de streaming
# au plus 2 fichiers par micro-batch
# Un flux de fichiers exige un schéma explicite : Spark ne peut pas l'inférer sur des fichiers
# qui n'existent pas encore. maxFilesPerTrigger borne le travail d'un micro-batch : sans cela
# un retard accumulé serait avalé d'un seul coup, au prix d'un micro-batch énorme.
stream_df = (
    spark.readStream
    .schema(schema_flux)
    .option("maxFilesPerTrigger", 2)
    .json(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame

Est un streaming DataFrame : True
Colonnes : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage']


In [28]:
# Ajout des colonnes calculées -- exactement comme en batch
# Les mêmes règles de nettoyage qu'au carnet 2 : c'est l'intérêt de Structured Streaming, une
# transformation écrite pour un DataFrame statique fonctionne à l'identique sur un flux.
stream_enrichi = (
    stream_df
    .withColumn("velos_total", F.col("velos_meca") + F.col("velos_elec"))
    .withColumn(
        "taux_occupation",
        # bornettes négatives ramenées à 0, capacité nulle -> NULL, résultat borné dans [0, 1]
        F.when(
            F.col("capacite") > 0,
            F.round(F.least(F.greatest(
                (F.col("capacite") - F.greatest(F.col("bornettes_libres"), F.lit(0))) / F.col("capacite"),
                F.lit(0.0)), F.lit(1.0)), 4),
        ),
    )
    .withColumn("est_vide", F.col("velos_total") == 0)
)
print("Colonnes après enrichissement :", stream_enrichi.columns)

Colonnes après enrichissement : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage', 'velos_total', 'taux_occupation', 'est_vide']


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [29]:
# Requête de streaming vers la console
# outputMode :
#   "append"  -- écrit uniquement les nouvelles lignes (défaut pour les flux sans agrégation)
#   "complete" -- réécrit l'intégralité du résultat à chaque batch
#   "update"  -- écrit uniquement les lignes qui ont changé
#
# La fonction start() lance le streaming

q_console = (
    stream_enrichi
    .select("station_id", "nom_station", "code_arr", "horodatage",
            "velos_total", "taux_occupation", "est_vide")
    .writeStream
    .format("console")                          # débogage uniquement : rien de persistant
    .outputMode("append")                       # sans agrégation, seules de nouvelles lignes arrivent
    .trigger(processingTime="5 seconds")        # un micro-batch toutes les 5 s
    .option("numRows", 10)                      # n'affiche que 10 lignes par batch
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "console"))
    .queryName("console_debug")
    .start()
)

# La requête tourne en arrière-plan.
# awaitTermination() bloquerait indéfiniment -- on attend juste quelques batchs.
time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(f"Batchs traités : {q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}")
q_console.stop()
print("Requête console arrêtée.")
# NB : la sortie « console » est écrite sur la sortie standard de la JVM (visible dans les logs
# du conteneur : `docker logs climacity-jupyter`), pas dans la cellule du carnet.

Statut de la requête : {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Batchs traités : 1376
Requête console arrêtée.


---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [30]:
from pyspark.sql.functions import window

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
df_fenetre_arr = (
    stream_enrichi
    # Watermark : Spark attend jusqu'à 5 min de retard sur l'horodatage maximal déjà vu avant
    # de fermer une fenêtre. Il est INDISPENSABLE en mode append : sans lui Spark ne saurait
    # jamais quand un résultat est définitif, donc ne l'émettrait jamais.
    .withWatermark("horodatage", "5 minutes")
    # window(col, 10 min, 2 min) = fenêtres de 10 min qui démarrent toutes les 2 min :
    # chaque relevé appartient donc à 5 fenêtres (10 / 2).
    .groupBy(window("horodatage", "10 minutes", "2 minutes"), "code_arr")
    .agg(
        F.sum("velos_total").alias("velos_disponibles"),
        F.approx_count_distinct("station_id").alias("nb_stations"),
        F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"),
    )
    # On « déplie » la structure window(start, end) en deux colonnes simples
    .select(
        F.col("window.start").alias("fenetre_debut"),
        F.col("window.end").alias("fenetre_fin"),
        "code_arr", "velos_disponibles", "nb_stations", "taux_moyen",
    )
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()

Schéma de la requête fenêtrée :
root
 |-- fenetre_debut: timestamp (nullable = true)
 |-- fenetre_fin: timestamp (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- velos_disponibles: long (nullable = true)
 |-- nb_stations: long (nullable = false)
 |-- taux_moyen: double (nullable = true)



In [31]:
# Écriture vers Delta Lake (sink delta) -- mode append
path_fenetres = str(CHEMIN_FENETRES)
q_fenetres = (
    df_fenetre_arr
    .writeStream
    .format("delta")                                   # sink transactionnel : chaque batch = 1 commit
    .outputMode("append")                              # une fenêtre n'est écrite qu'une fois, fermée
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "fenetres"))   # reprise après panne
    .queryName("fenetres_arrondissement")
    .trigger(processingTime="10 seconds")
    .start(path_fenetres)
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")

Requête démarrée : fenetres_arrondissement
ID               : 9cedd8ec-b51b-43b0-92dc-cb0d2faba553
En attente de 3 batchs...


Dernier progrès : {'id': '9cedd8ec-b51b-43b0-92dc-cb0d2faba553', 'runId': '98f9ef4d-1c48-4ff9-986c-ac692dc3ca02', 'name': 'fenetres_arrondissement', 'timestamp': '2026-09-21T22:43:00.000Z', 'batchId': 3, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'addBatch': 580, 'commitOffsets': 11, 'getBatch': 1, 'latestOffset': 1, 'queryPlanning': 12, 'triggerExecution': 616, 'walCommit': 10}, 'eventTime': {'watermark': '2021-01-01T02:05:00.000Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 210, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 0, 'numRowsRemoved': 210, 'allRemovalsTimeMs': 122, 'commitTimeMs': 116, 'memoryUsedBytes': 294848, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 8, 'numStateStoreInstances': 8, 'customMetrics': {'loadedMapCacheHitCount': 48, 'loadedMapCacheMissCount': 0, 'stateOnCurrentVersionSizeBytes': 142976}}], 'sources': [{'description': 'FileStreamSource[file:/home/jovyan/data/output/stream_in

In [32]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    # Lire les données au format delta, trier par début de fenêtre et en afficher 30
    (
        spark.read.format("delta").load(path_fenetres)
        .orderBy("fenetre_debut", "code_arr")
        .show(30, truncate=False)
    )
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")

+-------------------+-------------------+--------+-----------------+-----------+----------+
|fenetre_debut      |fenetre_fin        |code_arr|velos_disponibles|nb_stations|taux_moyen|
+-------------------+-------------------+--------+-----------------+-----------+----------+
|2020-12-31 23:10:00|2020-12-31 23:20:00|1       |468              |26         |0.6931    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|2       |347              |26         |0.4934    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|3       |271              |13         |0.6446    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|4       |458              |28         |0.6196    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|5       |623              |36         |0.5075    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|6       |481              |33         |0.4883    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|7       |717              |30         |0.6283    |
|2020-12-31 23:10:00|2020-12-31 23:20:00|8       |721              |52         |

---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [33]:
# Table d'alertes : schema
schema_alertes = StructType([
    StructField("station_id",   IntegerType(),   False),
    StructField("nom_station",  StringType(),    True),
    StructField("code_arr",     IntegerType(),   True),
    StructField("debut_rupture",TimestampType(), True),
    StructField("fin_rupture",  TimestampType(), True),
    StructField("duree_min",    IntegerType(),   True),
    StructField("ts_alerte",    TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
# Je crée la table à partir d'un DataFrame vide qui porte le schéma. Elle existe donc (avec son
# schéma figé) avant la première alerte, et je peux la lire sans erreur même si aucune
# alerte n'a encore eu lieu.
(
    spark.createDataFrame([], schema=schema_alertes)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(DELTA_ALERTES))
)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")

Table d'alertes initialisée : ../data/output/delta/alertes


In [34]:
from delta.tables import DeltaTable

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[int, dict] = {}
SEUIL_ALERTE_MIN = 10   # une rupture n'est signalée que si elle dure plus de 10 minutes

def cloturer_rupture(station_id: int, alertes: list) -> None:
    """Termine la rupture d'une station et l'ajoute à `alertes` si elle est assez longue.

    Args:
        station_id: Station dont la rupture s'achève.
        alertes: Liste (modifiée en place) des alertes du batch, au format du schéma
            ``schema_alertes``.

    Example:
        >>> etat_ruptures[1] = {"debut": t0, "fin": t0 + timedelta(minutes=15), "nom": "A", "arr": 5}
        >>> alertes = []; cloturer_rupture(1, alertes); len(alertes)
        1
    """
    rupture = etat_ruptures.pop(station_id)
    duree_min = (rupture["fin"] - rupture["debut"]).total_seconds() / 60
    if duree_min > SEUIL_ALERTE_MIN:
        alertes.append((
            station_id, rupture["nom"], rupture["arr"],
            rupture["debut"], rupture["fin"], int(round(duree_min)),
            datetime.now(timezone.utc).replace(tzinfo=None),
        ))


def traiter_batch_alertes(batch_df, batch_id: int) -> None :
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    # Sortir si `batch_df` est vide (rien à traiter, on évite un job Spark inutile)
    if batch_df.isEmpty():
        return

    # Agrégation au niveau station sur ce batch
    # On souhaite collecter :
    # - l'horodatage le plus ancien
    # - l'horodatage le plus récent
    # - le nombre d'observations où la station est vide
    # - le nombre total de stations
    # - le code de l'arrondissement
    # - le nom de la station
    # Un SEUL agrégat distribué calcule toutes ces valeurs ; seul le petit résultat (une ligne
    # par station, ~1 400) revient au driver avec collect().
    etat_batch = (
        batch_df
        .groupBy("station_id")
        .agg(
            F.min("horodatage").alias("ts_min"),
            F.max("horodatage").alias("ts_max"),
            F.sum(F.col("est_vide").cast("int")).alias("nb_vide"),
            F.count("*").alias("nb_releves"),
            F.first("code_arr").alias("code_arr"),
            F.first("nom_station").alias("nom_station"),
            # Premier / dernier instant où la station était vide dans ce batch, et son état
            # LE PLUS RÉCENT (max_by = valeur de est_vide au ts_max) : c'est ce dernier état
            # qui décide si la rupture continue ou se termine.
            F.min(F.when(F.col("est_vide"), F.col("horodatage"))).alias("debut_vide"),
            F.max(F.when(F.col("est_vide"), F.col("horodatage"))).alias("fin_vide"),
            F.max_by("est_vide", "horodatage").alias("derniere_vide"),
        )
        .collect()
    )

    alertes = []
    # Examiner les transitions :
    # - non vide -> vide
    # - vide -> non vide
    # Dans le premier cas, ajouter la les informations dela station dans `etat_rupture`.
    # Dans le second cas :
    # - calculer le temps pendant le quel la station a été en rupture
    # - si ce temps est > 10 minutes, ajouter une alerte avec les informations détailllées
    # - retirer la station de la liste `etat_rupture`
    # Durée d'une rupture = intervalle entre son premier et son dernier relevé VIDE : deux
    # relevés vides consécutifs (15 min d'écart) = 15 min de rupture avérée ; un seul relevé
    # vide = 0 min (peut-être un simple creux passager), donc pas d'alerte.
    for r in etat_batch:
        sid = r["station_id"]
        if sid in etat_ruptures:                       # la station était déjà en rupture
            if r["fin_vide"] is not None:              #   ... et elle l'est encore : on prolonge
                etat_ruptures[sid]["fin"] = r["fin_vide"]
            if not r["derniere_vide"]:                 #   vide -> non vide : la rupture s'achève
                cloturer_rupture(sid, alertes)
        elif r["debut_vide"] is not None:              # non vide -> vide : début d'une rupture
            etat_ruptures[sid] = {
                "debut": r["debut_vide"], "fin": r["fin_vide"],
                "nom": r["nom_station"], "arr": r["code_arr"],
            }
            if not r["derniere_vide"]:                 # rupture déjà terminée dans ce même batch
                cloturer_rupture(sid, alertes)

    # Si des alertes ont été enregistrées,
    # les sauvegarder au format `delta` dans DELTA_ALERTES
    # Laisser une trace dans les logs
    if alertes:
        (
            spark.createDataFrame(alertes, schema=schema_alertes)
            .write.format("delta").mode("append").save(str(DELTA_ALERTES))
        )
        journaliser(f"batch {batch_id} : {len(alertes)} alerte(s) de rupture écrite(s) "
                    f"({len(etat_ruptures)} station(s) actuellement en rupture)")
    else:
        journaliser(f"batch {batch_id} : aucune alerte ({len(etat_ruptures)} en rupture)")

In [35]:
# Lancement de la requête d'alerte
# - avec une tolérance de 5 minutes sur l'horodatage (cf. section suivante)
# - qui exécute la fonction `traiter_batch_alertes` pour chaque batch
# - définissant un `checkpoint` pour la sauvegarde de l'état en cas de panne
# - la requête s'appelle `q_alertes_rupture`
# - avec un intervalle de traitement de 10 ssecondes
q_alertes = (
    stream_enrichi
    # Sans agrégation, le watermark ne supprime rien : il ne fait que déclarer la tolérance
    # de 5 min, utile dès qu'on ajoutera une opération à état (jointure, agrégation).
    .withWatermark("horodatage", "5 minutes")
    .writeStream
    .foreachBatch(traiter_batch_alertes)        # notre fonction Python reçoit chaque micro-batch
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "alertes"))
    .queryName("q_alertes_rupture")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)

Requête d'alertes démarrée. En attente de données...


[LOG] batch 0 : aucune alerte (18 en rupture)


[LOG] batch 1 : 4 alerte(s) de rupture écrite(s) (18 station(s) actuellement en rupture)


[LOG] batch 2 : 2 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)


[LOG] batch 3 : 2 alerte(s) de rupture écrite(s) (13 station(s) actuellement en rupture)


[LOG] batch 4 : 1 alerte(s) de rupture écrite(s) (14 station(s) actuellement en rupture)


[LOG] batch 5 : 3 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)


[LOG] batch 6 : 2 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)



Alertes enregistrées : 14


+----------+-----------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|station_id|nom_station                        |code_arr|debut_rupture      |fin_rupture        |duree_min|ts_alerte                 |
+----------+-----------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|2005      |Place de Passy                     |16      |2020-12-31 23:19:00|2021-01-01 04:53:00|334      |2026-09-21 22:44:00.114901|
|1834      |Ménilmontant - Pelleport           |20      |2021-01-01 02:10:00|2021-01-01 04:53:00|163      |2026-09-21 22:44:00.114761|
|2223      |Saint-Jacques - Soufflot           |5       |2020-12-31 23:19:00|2021-01-01 04:02:00|283      |2026-09-21 22:43:50.141289|
|2254      |Solitaires - Place des Fêtes       |19      |2020-12-31 23:19:00|2021-01-01 04:02:00|283      |2026-09-21 22:43:50.141156|
|1933      |Pierre Larousse - Raymond Losserand|14     

---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [36]:
import json
from datetime import datetime as dt, timedelta   # alias : ne pas écraser `datetime` (utilisé par le logger)

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertroie surveillé par Spark.
# Chaque line est homogène à une observation (ou snapshot).
# N.B. Ajouter ne migne dans le fichier de logs
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.

    Raises:
        FileNotFoundError: Si le simulateur n'a encore produit aucun fichier.
    """
    # « Dans le passé » se mesure par rapport à l'INSTANT LE PLUS RÉCENT DU FLUX (le temps
    # simulé), pas par rapport à l'horloge : le simulateur rejoue janvier 2021, il ne serait
    # pas pertinent de comparer à aujourd'hui. Le dernier fichier émis donne cet instant.
    fichiers = sorted(STREAM_SOURCE_DIR.glob("velib_[0-9]*.json"))
    if not fichiers:
        raise FileNotFoundError("Aucun fichier du simulateur : impossible de dater les retards.")
    dernier = json.loads(fichiers[-1].read_text(encoding="utf-8"))
    ts_ref = max(
        dt.strptime(l["horodatage"], "%Y-%m-%dT%H:%M:%SZ") for l in dernier
    )
    ts_tardif = ts_ref - timedelta(minutes=retard_minutes)

    lignes = []
    for ligne in dernier[:nb_lignes]:
        tardive = dict(ligne)
        tardive["horodatage"] = ts_tardif.strftime("%Y-%m-%dT%H:%M:%SZ")
        lignes.append(tardive)

    # J'écris le fichier sous un nom temporaire puis je le renomme, comme le simulateur : Spark
    # ne lit ainsi jamais un fichier à moitié écrit.
    nom = STREAM_SOURCE_DIR / f"velib_tardif_{retard_minutes:03d}min_{time.time_ns()}.json"
    tmp = STREAM_SOURCE_DIR / f".tmp_{nom.name}"
    tmp.write_text(json.dumps(lignes), encoding="utf-8")
    tmp.rename(nom)
    journaliser(f"Injection de {len(lignes)} lignes en retard de {retard_minutes} min "
                f"(ts={ts_tardif:%Y-%m-%d %H:%M}, référence flux={ts_ref:%Y-%m-%d %H:%M})")

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(30)   # laisse le temps aux micro-batchs (10 s) de consommer le fichier

# Scénario 2 : retard important -- hors watermark -> les agrégations DEVRAIENT l'ignorer.
# NON CONFIRMÉ à l'exécution : le compteur numRowsDroppedByWatermark affiché plus bas reste à 0
# (voir README, § Limites) ; la théorie (fenêtre 10 min + watermark 5 min) prévoit l'écart.
# (le modèle indiquait 12 min, mais Spark n'écarte une ligne qu'une fois fermées TOUTES les
# fenêtres qui la contiennent : fenêtre 10 min + watermark 5 min, soit environ 15 min.
# 12 min de retard tombent encore dans une fenêtre ouverte, donc je prends 30 min.)
injecter_donnees_tardives(retard_minutes=30, nb_lignes=10)
time.sleep(30)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    # numRowsDroppedByWatermark : lignes écartées car trop en retard (requêtes à état seulement)
    # J'additionne sur toute la progression récente : le dernier micro-batch est souvent vide
    # et afficherait 0 même si Spark a écarté une ligne quelques secondes plus tôt.
    perdues = sum(
        (p.get("stateOperators") or [{}])[0].get("numRowsDroppedByWatermark", 0)
        for p in q.recentProgress
    ) if prog and prog.get("stateOperators") else "-"
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}"
          f"  -- lignes écartées par le watermark : {perdues}")
    if q.name == "fenetres_arrondissement" and prog:
        # Détail brut de l'opérateur à état : permet de lire soi-même les compteurs
        print("     détail stateOperators :", (prog.get("stateOperators") or [{}])[0])
        print("     watermark actuel      :", prog.get("eventTime", {}).get("watermark"))

[LOG] Injection de 10 lignes en retard de 3 min (ts=2021-01-01 05:18, référence flux=2021-01-01 05:21)


[LOG] Injection de 10 lignes en retard de 30 min (ts=2021-01-01 06:19, référence flux=2021-01-01 06:49)



Statut des requêtes actives :
  q_alertes_rupture              -- inputRows : 1377  -- lignes écartées par le watermark : -
  fenetres_arrondissement        -- inputRows : 0  -- lignes écartées par le watermark : 0
     détail stateOperators : {'operatorName': 'stateStoreSave', 'numRowsTotal': 210, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 0, 'numRowsRemoved': 210, 'allRemovalsTimeMs': 119, 'commitTimeMs': 115, 'memoryUsedBytes': 295360, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 8, 'numStateStoreInstances': 8, 'customMetrics': {'loadedMapCacheHitCount': 240, 'loadedMapCacheMissCount': 0, 'stateOnCurrentVersionSizeBytes': 143488}}
     watermark actuel      : 2021-01-01T07:49:00.000Z


In [37]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
# Sans 3e argument, window() est BASCULANTE : les fenêtres se suivent sans se chevaucher, donc
# chaque relevé tombe dans une seule fenêtre. Le mode "update" n'émet que les fenêtres dont le
# maximum vient de changer, ce qui convient à une agrégation qu'on veut voir évoluer en direct
# (contrairement à "append", qui attendrait la fermeture de la fenêtre).
q_exercice = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(window("horodatage", "15 minutes"), "code_arr")
    .agg(F.max("taux_occupation").alias("taux_max"))
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "exercice"))
    .queryName("exercice_max_15min")
    .trigger(processingTime="10 seconds")
    .start()
)
time.sleep(20)
print(f"Requête « {q_exercice.name} » active : {q_exercice.isActive}")

Requête « exercice_max_15min » active : True


---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [38]:
# Arrêt de toutes les requêtes actives
actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

# Arrêter proprement les requêtes et reseigner le fichier logs
# stop() attend la fin du micro-batch en cours : le checkpoint reste cohérent (pas de perte).
for q in actives:
    q.stop()
    journaliser(f"Requête arrêtée : {q.name}")

# Le simulateur est un processus à part : on l'arrête aussi (sinon il continuerait à produire).
if SIM_PROC is not None and SIM_PROC.poll() is None:
    SIM_PROC.terminate()
    SIM_PROC.wait(timeout=10)
    journaliser("Simulateur arrêté")

print("\nToutes les requêtes de streaming sont arrêtées.")
print(f"Journal complet : {LOG_STREAM}")

Requêtes actives à arrêter : ['q_alertes_rupture', 'exercice_max_15min', 'fenetres_arrondissement']
[LOG] Requête arrêtée : q_alertes_rupture
[LOG] Requête arrêtée : exercice_max_15min
[LOG] Requête arrêtée : fenetres_arrondissement
[LOG] Simulateur arrêté

Toutes les requêtes de streaming sont arrêtées.
Journal complet : ../data/output/logs/streaming.log


In [39]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


=== Résultats finaux ===

-- Fenêtres arrondissement --


  3173 fenêtres enregistrées


+-------------------+-------------------+--------+-----------------+-----------+----------+
|fenetre_debut      |fenetre_fin        |code_arr|velos_disponibles|nb_stations|taux_moyen|
+-------------------+-------------------+--------+-----------------+-----------+----------+
|2021-01-01 08:14:00|2021-01-01 08:24:00|9       |476              |45         |0.3928    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|13      |1048             |67         |0.4744    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|21      |1089             |88         |0.4169    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|5       |627              |36         |0.5211    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|27      |51               |5          |0.429     |
|2021-01-01 08:14:00|2021-01-01 08:24:00|42      |403              |32         |0.4162    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|10      |611              |53         |0.4297    |
|2021-01-01 08:14:00|2021-01-01 08:24:00|48      |48               |10         |

  35 alertes enregistrées


+----------+-------------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|station_id|nom_station                          |code_arr|debut_rupture      |fin_rupture        |duree_min|ts_alerte                 |
+----------+-------------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|2203      |Saint-Amand - Labrouste              |15      |2021-01-01 06:49:00|2021-01-01 08:52:00|123      |2026-09-21 22:45:20.117556|
|1091      |Belleville -  Pré Saint-Gervais      |19      |2021-01-01 07:20:00|2021-01-01 08:52:00|92       |2026-09-21 22:45:20.117503|
|1382      |Faustin Hélie - Desbordes-Valmore    |16      |2021-01-01 02:10:00|2021-01-01 08:52:00|402      |2026-09-21 22:45:20.117444|
|2223      |Saint-Jacques - Soufflot             |5       |2021-01-01 07:20:00|2021-01-01 08:15:00|55       |2026-09-21 22:45:10.132005|
|2287      |Square du Carrefour de l'Insu

### Extrait du journal d'exécution

In [40]:
# Les dernières lignes du fichier de logs : on y retrouve les alertes, l'injection de données
# tardives et l'arrêt des requêtes.
print("".join(LOG_STREAM.read_text(encoding="utf-8").splitlines(keepends=True)[-15:]))

2026-09-21 22:43:50,692 [INFO] batch 5 : 3 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)
2026-09-21 22:44:00,679 [INFO] batch 6 : 2 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)
2026-09-21 22:44:08,572 [INFO] Injection de 10 lignes en retard de 3 min (ts=2021-01-01 05:18, référence flux=2021-01-01 05:21)
2026-09-21 22:44:10,801 [INFO] batch 7 : 3 alerte(s) de rupture écrite(s) (15 station(s) actuellement en rupture)
2026-09-21 22:44:20,659 [INFO] batch 8 : 3 alerte(s) de rupture écrite(s) (12 station(s) actuellement en rupture)
2026-09-21 22:44:31,226 [INFO] batch 9 : 3 alerte(s) de rupture écrite(s) (11 station(s) actuellement en rupture)
2026-09-21 22:44:38,580 [INFO] Injection de 10 lignes en retard de 30 min (ts=2021-01-01 06:19, référence flux=2021-01-01 06:49)
2026-09-21 22:44:40,849 [INFO] batch 10 : 1 alerte(s) de rupture écrite(s) (13 station(s) actuellement en rupture)
2026-09-21 22:44:51,309 [INFO] batch 11 : 2 alerte(s) de r

---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [41]:
spark.stop()
print("SparkSession arrêtée. À demain !")


SparkSession arrêtée. À demain !
